# P2 — Strategy schematic on a synthetic z-score

Goal: a single self-contained figure that makes the *mechanical* difference between
`Baseline`, `AR` (hard-kill) and `MS_AR` (dynamic gearbox) obvious in one glance.

No real data — we hand-build a synthetic z-score with a clearly visible 'danger phase'
in the middle of the series so every rule reacts to the same shocks.

Output:  `code/plots/eda/strategy_schematic.pdf`

In [ ]:
import os, sys
if os.path.exists('/content'):
    os.system('curl -sL https://raw.githubusercontent.com/egil10/stk-mat2011/main/code/scripts/colab.py -o /content/colab.py')
    sys.path.insert(0, '/content')
    from colab import setup; setup('code/visualisations')
else:
    sys.path.insert(0, '../scripts')

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from plotting import apply_econ_style, ECON, save_figure_pdf
apply_econ_style()

In [ ]:
# ── synthetic z-score: calm → drift-with-elevated-noise → calm ──────
rng = np.random.default_rng(42)
n   = 1000
t   = np.arange(n)

z = np.zeros(n)
for i in range(1, n):
    z[i] = 0.70 * z[i-1] + rng.normal(0, 0.65)

lo, hi = 350, 650
z[lo:hi] += np.linspace(0, 2.6, hi - lo)            # drift bias
z[lo:hi] += rng.normal(0, 0.55, hi - lo)            # extra vol

# Synthetic Danger probability (with realistic ramp-up / ramp-down)
danger = np.zeros(n)
ramp_in  = np.clip(np.linspace(-0.2, 1.0, 40), 0, 1)
ramp_out = ramp_in[::-1]
danger[lo-20: lo+20] = ramp_in * 0.92
danger[lo+20: hi-20] = 0.92
danger[hi-20: hi+20] = ramp_out * 0.92
mr_prob = 1.0 - danger

# Strategy parameters (same as in code/scripts/month.py)
z_q, z_v, exit_z, delta = 1.3, 2.5, 0.0, 0.30

In [ ]:
def gen_positions(z_arr, entry_arr, exit_z, gate):
    """Numba-free mirror of backtester._generate_positions."""
    pos = np.zeros(len(z_arr))
    curr = 0.0
    for i in range(len(z_arr)):
        if curr != 0 and not gate[i]:
            curr = 0.0                                     # panic kill
        if curr == 0:
            if gate[i]:
                if z_arr[i] < -entry_arr[i]: curr = 1.0
                elif z_arr[i] >  entry_arr[i]: curr = -1.0
        elif curr ==  1 and z_arr[i] >= -exit_z: curr = 0.0
        elif curr == -1 and z_arr[i] <=  exit_z: curr = 0.0
        pos[i] = curr
    return pos

gate_open = np.ones(n, dtype=bool)
gate_ar   = mr_prob >= (1.0 - delta)
entry_st  = np.full(n, z_q)
entry_dyn = mr_prob * z_q + danger * z_v

pos_base = gen_positions(z, entry_st,  exit_z, gate_open)
pos_ar   = gen_positions(z, entry_st,  exit_z, gate_ar)
pos_msar = gen_positions(z, entry_dyn, exit_z, gate_open)

In [ ]:
fig, axes = plt.subplots(
    4, 1, figsize=(12.5, 10.5), sharex=True,
    gridspec_kw={'height_ratios': [2.2, 2.2, 2.2, 0.7]},
)

def shade_pos(ax, pos):
    long_mask  = pos ==  1
    short_mask = pos == -1
    ax.fill_between(t, -10, 10, where=long_mask,  color=ECON['green'], alpha=0.12, lw=0, step='post')
    ax.fill_between(t, -10, 10, where=short_mask, color=ECON['red'],   alpha=0.12, lw=0, step='post')

# Panel 1 — Baseline
ax = axes[0]
shade_pos(ax, pos_base)
ax.plot(t, z, color=ECON['navy'], lw=0.8)
for v in (z_q, -z_q):
    ax.axhline(v, color=ECON['grey'], lw=0.8, ls='--')
ax.axhline(0, color=ECON['lt_grey'], lw=0.5)
ax.set_ylim(-4.2, 4.2); ax.set_ylabel('z-score')
ax.set_title('Baseline  —  constant ±z_q band, no regime gate', loc='left', fontweight='bold')

# Panel 2 — AR (hard kill)
ax = axes[1]
ax.fill_between(t, -10, 10, where=~gate_ar, color=ECON['lt_grey'], alpha=0.45, lw=0, step='post')
shade_pos(ax, pos_ar)
ax.plot(t, z, color=ECON['navy'], lw=0.8)
for v in (z_q, -z_q):
    ax.axhline(v, color=ECON['grey'], lw=0.8, ls='--')
ax.axhline(0, color=ECON['lt_grey'], lw=0.5)
ax.set_ylim(-4.2, 4.2); ax.set_ylabel('z-score')
ax.set_title('AR  —  constant ±z_q + hard kill when MR_Prob < 1−δ (grey)', loc='left', fontweight='bold')

# Panel 3 — MS_AR (dynamic band)
ax = axes[2]
shade_pos(ax, pos_msar)
ax.plot(t, z, color=ECON['navy'], lw=0.8, label='z-score')
ax.plot(t,  entry_dyn, color=ECON['MS_AR'], lw=1.1, label='±z_q^{eff} (dynamic)')
ax.plot(t, -entry_dyn, color=ECON['MS_AR'], lw=1.1)
ax.axhline(0, color=ECON['lt_grey'], lw=0.5)
ax.set_ylim(-4.2, 4.2); ax.set_ylabel('z-score')
ax.set_title('MS_AR  —  dynamic ±z_q^{eff} band (purple), no kill', loc='left', fontweight='bold')
ax.legend(loc='lower left', frameon=False, fontsize=8)

# Panel 4 — Danger probability strip
ax = axes[3]
ax.fill_between(t, 0, danger, color=ECON['red'], alpha=0.55, lw=0)
ax.axhline(delta, color=ECON['navy'], lw=0.8, ls=':')
ax.text(8, delta + 0.04, f'δ = {delta:.2f}', fontsize=8, color=ECON['navy'])
ax.set_ylim(0, 1); ax.set_ylabel('Danger\nProb', fontsize=9)
ax.set_xlabel('synthetic bar index')

legend_elems = [
    Patch(facecolor=ECON['green'],   alpha=0.35, label='Long  (z < −z_q)'),
    Patch(facecolor=ECON['red'],     alpha=0.35, label='Short (z > +z_q)'),
    Patch(facecolor=ECON['lt_grey'], alpha=0.55, label='AR kill zone'),
]
fig.legend(handles=legend_elems, loc='upper center', ncol=3,
           bbox_to_anchor=(0.5, 0.995), frameon=False, fontsize=10)
fig.suptitle('How the three strategies translate the same z-score into positions',
             y=1.025, fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.965])
save_figure_pdf(fig, 'strategy_schematic.pdf', pdf_dir='../plots/eda', enabled=True, verbose=True)
plt.show()